In [ ]:
import config as cfg
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import src.loading as ld
import src.features as ftrs
import src.normalise as norm
import src.evaluation as evl
from sklearn.base import clone
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import LeaveOneGroupOut
from scipy.stats import wilcoxon

In [ ]:
df = ld.load_frames()
df_n = norm.normalise_d(df)
feature_table = ftrs.build_feature_table(df_n)

feature_names = feature_table.columns.tolist()
X = feature_table.to_numpy()
y = feature_table.index.get_level_values("emotion").to_numpy()
groups = feature_table.index.get_level_values("actor").to_numpy()

### Helper Functions

In [ ]:
cv_loso = LeaveOneGroupOut()
K_GRID = [5, 10, 15]

def selection_pipeline(selector=None):
    steps = [("scaler", StandardScaler())]
    if selector is not None:
        steps.append(("select", selector))
    steps.append(("svc", SVC(kernel="rbf", C=cfg.C_FIXED)))
    return Pipeline(steps)

def results_table(names):
    return pd.DataFrame({name: {"macro F1 mean": loso_results[name]["macro_f1"].mean(),
                                "macro F1 std": loso_results[name]["macro_f1"].std(),
                                "accuracy mean": loso_results[name]["accuracy"].mean()}
                         for name in names}).T.round(4)

loso_results = {"All (55)": evl.evaluation(X, y, cv_loso, selection_pipeline(), groups)[0]}